# Aggregate fio read bytes and throughput across ranks

Sums the per-rank `read.io_bytes` and bandwidth from the fio JSON output files
(one file per MPI rank). Rank files may have a non-JSON header line (an MPI/PBS
error message), so parsing starts at the first `{`.

Two aggregate throughput numbers are reported:
- **Sum of per-rank bw**: each rank's own `bw_bytes` summed (optimistic; ignores rank skew).
- **Total bytes / max runtime**: wall-clock view, bounded by the slowest rank.


In [1]:
import json
import glob
from pathlib import Path

# Folder containing one fio JSON file per rank
RESULT_DIR = Path("read_1tib_n2_ppn32_bs2m_iod16_8650075")

files = sorted(RESULT_DIR.glob("*.json"))
print(f"{len(files)} rank files")

64 rank files


In [2]:
def load_fio_json(path):
    """Parse a fio JSON file, skipping any non-JSON header lines."""
    txt = Path(path).read_text()
    return json.loads(txt[txt.index("{"):])

per_rank = []
for fp in files:
    d = load_fio_json(fp)
    per_rank.append({
        "file": fp.name,
        "io_bytes": sum(j["read"]["io_bytes"] for j in d["jobs"]),
        "bw_bytes": sum(j["read"]["bw_bytes"] for j in d["jobs"]),
        "runtime_ms": max(j["read"]["runtime"] for j in d["jobs"]),
    })

In [3]:
GiB = 1024**3

total_bytes = sum(r["io_bytes"] for r in per_rank)
total_bw_bytes = sum(r["bw_bytes"] for r in per_rank)
max_rt_s = max(r["runtime_ms"] for r in per_rank) / 1000
min_rt_s = min(r["runtime_ms"] for r in per_rank) / 1000

print(f"Aggregated read bytes    : {total_bytes} B = {total_bytes/GiB:.2f} GiB = {total_bytes/1024**4:.4f} TiB")
print(f"Runtime (min-max)        : {min_rt_s:.1f} - {max_rt_s:.1f} s")
print(f"Sum of per-rank bw       : {total_bw_bytes/GiB:.2f} GiB/s = {total_bw_bytes/1e9:.2f} GB/s")
print(f"Total bytes / max runtime: {total_bytes/max_rt_s/GiB:.2f} GiB/s = {total_bytes/max_rt_s/1e9:.2f} GB/s")

Aggregated read bytes    : 1099511627776 B = 1024.00 GiB = 1.0000 TiB
Runtime (min-max)        : 10.3 - 13.9 s
Sum of per-rank bw       : 84.41 GiB/s = 90.63 GB/s
Total bytes / max runtime: 73.82 GiB/s = 79.27 GB/s


## Comparison: single-process run (`fio-read1TiB-seqread-fs16g.json`)

A single fio invocation with `numjobs=64`, `size=16gb` per job, `group_reporting=1`
(one JSON file for the whole run), vs. the 32-rank MPI run above.

In [4]:
SINGLE_RUN = Path("fio-read1TiB-seqread-fs16g.json")

d = load_fio_json(SINGLE_RUN)
single = {
    "io_bytes": sum(j["read"]["io_bytes"] for j in d["jobs"]),
    "bw_bytes": sum(j["read"]["bw_bytes"] for j in d["jobs"]),
    "runtime_s": max(j["read"]["runtime"] for j in d["jobs"]) / 1000,
}
print(f"read bytes : {single['io_bytes']} B = {single['io_bytes']/GiB:.2f} GiB = {single['io_bytes']/1024**4:.4f} TiB")
print(f"runtime    : {single['runtime_s']:.1f} s")
print(f"bandwidth  : {single['bw_bytes']/GiB:.2f} GiB/s = {single['bw_bytes']/1e9:.2f} GB/s")

read bytes : 1099511627776 B = 1024.00 GiB = 1.0000 TiB
runtime    : 38.1 s
bandwidth  : 26.85 GiB/s = 28.83 GB/s


In [6]:
rows = [
    ("MPI run", total_bytes, max_rt_s, total_bw_bytes, total_bytes / max_rt_s),
    ("single-process run", single["io_bytes"], single["runtime_s"], single["bw_bytes"], single["io_bytes"] / single["runtime_s"]),
]

hdr = f"{'run':<20} {'read GiB':>10} {'runtime s':>10} {'sum bw GiB/s':>13} {'wall bw GiB/s':>14}"
print(hdr)
print("-" * len(hdr))
for name, b, rt, bw, wall in rows:
    print(f"{name:<20} {b/GiB:>10.2f} {rt:>10.1f} {bw/GiB:>13.2f} {wall/GiB:>14.2f}")

ratio = (total_bytes / max_rt_s) / (single["io_bytes"] / single["runtime_s"])
print(f"\nMPI run wall-clock bandwidth is {ratio:.2f}x the single-process run")

run                    read GiB  runtime s  sum bw GiB/s  wall bw GiB/s
-----------------------------------------------------------------------
MPI run                 1024.00       13.9         84.41          73.82
single-process run      1024.00       38.1         26.85          26.85

MPI run wall-clock bandwidth is 2.75x the single-process run


## Iteration-by-iteration: repeated-read job (`read_1tib_n2_ppn32_bs2m_iod16_8651348`)

This job repeats the 64-rank read pass NITERS times (see `qsub_fio_read_1tib.qsub`);
each pass writes 64 rank files labelled `..._iterNN_<epoch>_rankNNN.json`.
Files are grouped by their `iterNN` tag and each iteration is aggregated the
same way as above (sum of per-rank bw, and total bytes / max runtime).

In [ ]:
import re
from collections import defaultdict

# All raw rank JSONs live in the read_1tib_n2_ppn32_bs2m_iod16_* job folders.
# Group by (job folder, iterNN tag); files without an iter tag (the older
# single-pass job) fall into one "-" group for that folder.
runs = defaultdict(list)  # (job, iter_tag) -> [rank json paths]
for fp in sorted(Path(".").glob("read_1tib_n2_ppn32_bs2m_iod16_*/*.json")):
    m = re.search(r"iter(\d+)", fp.name)
    runs[(fp.parent.name, m.group(1) if m else "-")].append(fp)

def aggregate(paths):
    """Aggregate one pass (a set of per-rank fio JSONs) -> summary dict."""
    io_b = bw_b = 0
    rts = []
    for fp in paths:
        d = load_fio_json(fp)
        io_b += sum(j["read"]["io_bytes"] for j in d["jobs"])
        bw_b += sum(j["read"]["bw_bytes"] for j in d["jobs"])
        rts.append(max(j["read"]["runtime"] for j in d["jobs"]) / 1000)
    return {"nranks": len(paths), "io_bytes": io_b, "bw_bytes": bw_b,
            "rt_min_s": min(rts), "rt_max_s": max(rts),
            "wall_bw": io_b / max(rts)}

per_iter = {key: aggregate(paths) for key, paths in sorted(runs.items())}

hdr = (f"{'job':<42} {'iter':>4} {'ranks':>5} {'read GiB':>9} "
       f"{'rt min-max s':>13} {'sum bw GiB/s':>13} {'wall bw GiB/s':>14}")
print(hdr)
print("-" * len(hdr))
for (job, it), a in per_iter.items():
    print(f"{job:<42} {it:>4} {a['nranks']:>5} {a['io_bytes']/GiB:>9.2f} "
          f"{a['rt_min_s']:>6.1f}-{a['rt_max_s']:<6.1f} "
          f"{a['bw_bytes']/GiB:>13.2f} {a['wall_bw']/GiB:>14.2f}")

In [ ]:
import statistics as st

# Stability across the repeated passes (iter-tagged runs only)
w = [a["wall_bw"] / GiB for (job, it), a in per_iter.items() if it != "-"]
print(f"wall-clock bw over {len(w)} iterations:")
print(f"  mean {st.mean(w):.2f}  stdev {st.stdev(w):.2f}  "
      f"min {min(w):.2f}  max {max(w):.2f} GiB/s")

In [ ]:
import matplotlib.pyplot as plt

# Per-rank bandwidth for every run: the single-pass job (8650075) plus the
# 9 iterations of the repeated job (8651348) -> 10 groups of 64 ranks.
labels, groups = [], []
for (job, it), paths in sorted(runs.items()):
    labels.append("single\npass" if it == "-" else f"iter\n{it}")
    groups.append([sum(j["read"]["bw_bytes"] for j in load_fio_json(fp)["jobs"]) / GiB
                   for fp in paths])

fig, ax = plt.subplots(figsize=(10, 4.5))
bp = ax.boxplot(groups, patch_artist=True, widths=0.55,
                medianprops=dict(color="#104281", linewidth=2),
                boxprops=dict(facecolor="#9ec5f4", edgecolor="#256abf", linewidth=1),
                whiskerprops=dict(color="#256abf", linewidth=1),
                capprops=dict(color="#256abf", linewidth=1),
                flierprops=dict(marker="o", markersize=4,
                                markerfacecolor="#6d7683", markeredgecolor="none"))
ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels)

ax.set_ylabel("per-rank read bandwidth (GiB/s)")
ax.set_title("fio 1 TiB read: per-rank bandwidth by run (64 ranks each)")
ax.yaxis.grid(True, color="#d5dae2", linewidth=0.75)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

## Random read: repeated-read job (`randread_1tib_n2_ppn32_bs2m_iod16_*`)

Same analysis as the sequential read above, for the `PATTERN=randread` job
(10 iterations × 64 ranks): per-iteration aggregates, stability stats, and a
per-rank bandwidth box plot grouped by iteration.

In [ ]:
# Group the randread rank files by (job folder, iterNN tag), aggregate each
# iteration with the same aggregate() as the sequential-read section.
rand_runs = defaultdict(list)
for fp in sorted(Path(".").glob("randread_1tib_n2_ppn32_bs2m_iod16_*/*.json")):
    m = re.search(r"iter(\d+)", fp.name)
    rand_runs[(fp.parent.name, m.group(1) if m else "-")].append(fp)

rand_per_iter = {key: aggregate(paths) for key, paths in sorted(rand_runs.items())}

hdr = (f"{'job':<46} {'iter':>4} {'ranks':>5} {'read GiB':>9} "
       f"{'rt min-max s':>13} {'sum bw GiB/s':>13} {'wall bw GiB/s':>14}")
print(hdr)
print("-" * len(hdr))
for (job, it), a in rand_per_iter.items():
    print(f"{job:<46} {it:>4} {a['nranks']:>5} {a['io_bytes']/GiB:>9.2f} "
          f"{a['rt_min_s']:>6.1f}-{a['rt_max_s']:<6.1f} "
          f"{a['bw_bytes']/GiB:>13.2f} {a['wall_bw']/GiB:>14.2f}")

w = [a["wall_bw"] / GiB for a in rand_per_iter.values()]
print(f"\nwall-clock bw over {len(w)} iterations:")
print(f"  mean {st.mean(w):.2f}  stdev {st.stdev(w):.2f}  "
      f"min {min(w):.2f}  max {max(w):.2f} GiB/s")

In [ ]:
# Box plot: per-rank randread bandwidth, one box per iteration (64 ranks each)
rand_labels, rand_groups = [], []
for (job, it), paths in sorted(rand_runs.items()):
    rand_labels.append(f"iter\n{it}")
    rand_groups.append([sum(j["read"]["bw_bytes"] for j in load_fio_json(fp)["jobs"]) / GiB
                        for fp in paths])

fig, ax = plt.subplots(figsize=(10, 4.5))
bp = ax.boxplot(rand_groups, patch_artist=True, widths=0.55,
                medianprops=dict(color="#104281", linewidth=2),
                boxprops=dict(facecolor="#9ec5f4", edgecolor="#256abf", linewidth=1),
                whiskerprops=dict(color="#256abf", linewidth=1),
                capprops=dict(color="#256abf", linewidth=1),
                flierprops=dict(marker="o", markersize=4,
                                markerfacecolor="#6d7683", markeredgecolor="none"))
ax.set_xticks(range(1, len(rand_labels) + 1))
ax.set_xticklabels(rand_labels)

ax.set_ylabel("per-rank read bandwidth (GiB/s)")
ax.set_title("fio 1 TiB randread: per-rank bandwidth by run (64 ranks each)")
ax.yaxis.grid(True, color="#d5dae2", linewidth=0.75)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()